# First model: Basic Gaussian Mixture

While the end-goal of this project is to have a functioning Hidden Markov Model to capture the temporal behaviour of our crops, it is first necessary for us to establish a baseline model operating under the assumption of statistical independence. The inherent problem of this project is unsupervised, therefore we consider a range of unsupervised models, starting off with clustering. The approach of fitting a GMM in this notebook answers the question: **Are conditions behaving abnormally right now?**

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook", palette="deep")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.mixture import GaussianMixture 
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

from models import NDVIClimatologyGAM, LikelihoodGMM, ForecastingGMM, ForecastingGMMSelector
from data import utils

## Load Data

In [3]:
loader = DatasetLoader(raw_data_dir="../data/raw/", engineered_data_dir="../data/processed/")
non_spectral_df = loader.non_spectral()
grouped_spectral_df = loader.grouped_spectral()
master_df = utils.merge_with_master(grouped_spectral_df, non_spectral_df)


In [4]:
# Engineered features
eng_non_spectral_df  = pd.read_csv("../data/processed/eng_non_spectral_df.csv", index_col=0)
eng_non_spectral_df["Timestamp"] = pd.to_datetime(eng_non_spectral_df["Timestamp"])
eng_non_spectral_df["in_season"] = eng_non_spectral_df["in_season"].astype('category')
eng_non_spectral_df["Growth_Stage"] = eng_non_spectral_df["Growth_Stage"].astype('category')


In [7]:
# Rolling statistics and lagged variables
roll_lag_df = pd.read_csv("../data/processed/roll_df.csv", index_col=0)
roll_lag_df["Timestamp"] = pd.to_datetime(roll_lag_df["Timestamp"])

In [8]:
non_spectral_df = non_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)

grouped_spectral_df = grouped_spectral_df.merge(
    eng_non_spectral_df[["Timestamp", "Ag_year", "shifted_doy"]],
    on="Timestamp",
    how="left"
)



### Extracting Anomaly Data

An immediate answer to the question "Why fit a GMM?" is apparent here. Our anomly years found in the EDA process are _not_ a sequential set (2018-2019, 2023, maybe 2024). This means there will be a time gap between the training years. In a **sequential** model like an HMM, this would be a problem, but a GMM makes the assumption of statistical independence, so the gap doesn't matter. 

In [9]:
builder = DatasetBuilder(init_df=test_df[eng_features + non_spectral_features])

gmm_df = (
    builder
    .merge_spectral(grouped_spectral_df[spectral_features])
    .interpolate(val_columns=spectral_features[1:], dropna=True)
    .build()
)

norm_df, anom_df = builder.split_by_growing_season(2018, 2019, 2023, 2024)

### Interpolation

We present the first issue of our data using a simple merge between the non-spectral and spectral datasets.

In [ ]:
master_raw_df = grouped_spectral_df.merge(non_spectral_df, on="Timestamp" ,how='right')
master_raw_df.head(7)

You will quickly notice that our spectral indices contain lots of `NaN` observations. This is due to the almost-weekly passes of the Sentinel-2 mission, as well as cloud-masking applied in our pre-processing steps. Luckily, NDVI follows a pretty general pattern and is not prone to extreme oscilations from one moment to the next. It is therefore not a bad idea to use the built-in **Cubic Hermit Spline** wrapper to `scipy` in `pandas` to account for these unobserved images. 

Note we also use **backfilling** to account for the missing observations at the start of our datasets, as the spectral indices are unlikely to vary much in such a short period.

In [ ]:
builder.plot_interpolate(data=grouped_spectral_df, x="Timestamp", y="NDVI_mean")

---

## Full Model, no lagged variables or rolling statistics

As a baseline, we fit a GMM on all our engineered and remaining non-spectral variables. The log likelihood of the training sample will give us a good indication of how well the model fit to the data. Note that, since GMM's assume _continuous_ variables, we do not explicitly make use of the `in_season` or `Growth-Stage` features. We will operate under the assumption that this information is baked into `cummulative_GDD`.

In [ ]:
eng_features = [
    # Timestamp for merging
    'Timestamp',

    # Date features for plotting
    "shifted_doy",
    "Ag_year",

    # engineered features
    'cumulative_GDD', 
    'root_weighted_soil_moisture', 
    #'VW_PC1',  
    #'Shallow_mean',
    'ST_PC1',

    # Categoricals not included
    #'Growth_Stage',
    'in_season',
]

non_spectral_features = [
    'Timestamp',
    'temperature_2m',
    #'temperature_2m_min',
    #'temperature_2m_max', 
    #'surface_solar_radiation_downwards_sum',
    'precipitation',
    'volumetric_soil_water_layer_3',
    #'volumetric_soil_water_layer_4',
    'soil_temperature_level_4'
]


In [ ]:
train_mask = norm_df["Ag_year"] < 2022
test_mask = norm_df["Ag_year"] >= 2022
passthrough_cols = ["doy_sin", "doy_cos"]
drop_cols = ["Timestamp", "shifted_doy", "doy", "GDD", "in_season", "Ag_year"]

scaler_splitter = DatasetScalerSplitter(
    train_mask=train_mask, 
    test_mask=test_mask, 
    passthrough_cols=passthrough_cols,
    drop_cols=drop_cols
)

scaler_splitter.fit(norm_df)

X_train_df, X_test_df = scaler_splitter.train_test_split_transform(norm_df)
X_anom_df = scaler_splitter.transform(anom_df)

In [ ]:
gmm = LikelihoodGMM(
    n_components=17, 
    covariance_type='full',
    reg_covar=1e-03,
    n_init=10
)
gmm.cv_fit(X_train_df)

In [ ]:
gmm.plot_bic_curve()

In [ ]:
elbow_gmm = LikelihoodGMM(
    n_components=9, 
    covariance_type=gmm.covariance_type,
    reg_covar=gmm.reg_covar,
    n_init=gmm.n_init
)
elbow_gmm.fit(X_train_df)

In [ ]:
print(f"{'':<35}{'Lowest BIC model':<25}{'Elbow Model'}")
print("-" * 80)
print(f"{'Test data log-likelihood':<35}{gmm.score(X_test_df):.2f}{'':<18}{elbow_gmm.score(X_test_df):.2f}")
print(f"{'Anomaly data log-likelihood':<35}{gmm.score(X_anom_df):.2f}{'':<18}{elbow_gmm.score(X_anom_df):.2f}")

In [ ]:
gmm.plot_likelihood_kde(X_test_df, X_anom_df, labels=["Test Data", "Anomaly Data"])

---

## Rolling and Lagged variables

While our first model does have pretty surprising performance, it does operate under two assumptions violated by the data:
1. **Independence between observations**.
2. **No temporal behaviour of variables**: The baseline GMM does not in any way use _past_ environmental data for _current_ observations during its clustering. This is a bad strategy for time series agricultural data because, for example, rain will not have a meaningful effect on a crop until a few days _after_ it has occurred. Similarly, a single day's drought will not put a plant under stress, but rather an _accumulation_ of days of drought.

While we cannot address the independence assumption since that is a defining characteristic of a GMM, we can include rolled statistics and lagging variables to force the model to learn on temporal data.

In [ ]:
roll_lag_features = [
    "Timestamp",
    
    # Temperature
    "temperature_2m_7D_mean",
    #"temperature_2m_30D_mean",

    # Solar radiation
    "surface_solar_radiation_downwards_sum_7D_mean",
    #"surface_solar_radiation_downwards_sum_30D_sum",

    # Precipitation
    "precipitation_7D_sum",
    #"precipitation_30D_sum",

    # Root-zone soil moisture (linear comb of first 3 volumetric soil water layers)
    #"root_weighted_soil_moisture_7D_mean",
    #"root_weighted_soil_moisture_30D_mean",

    # Volumetric soil water layers 1 and 2 (shallow) (first PC)
    #"VW_PC1_7D_mean",
    #"VW_PC1_30D_mean",

    # Soil temperatre levels 1-3 (shallow) PCs
    #"ST_PC1_7D_mean",
    #"ST_PC1_30D_mean",


    # Deep volumetric Soil Water Layers 3-4
    #"volumetric_soil_water_layer_3_7D_mean",
    "volumetric_soil_water_layer_3_30D_mean",
    #"volumetric_soil_water_layer_4_7D_mean",
    #"volumetric_soil_water_layer_4_30D_mean",
    

    # Deep soil temperature level 4
    #"soil_temperature_level_4_7D_mean",
    "soil_temperature_level_4_30D_mean",

]

eng_features = [
    # Date/time variables 
    'Timestamp',
    "shifted_doy",
    #"doy",
    "Ag_year",

    
    'cumulative_GDD', 
    'root_weighted_soil_moisture', 
    #'VW_PC1', 
    #'Shallow_mean',
    'ST_PC1',

    # Categoricals not valid for GMM
    #'in_season', 
    #'Growth_Stage',
    
]

non_spectral_features = [
    'Timestamp',
    #'temperature_2m',
    #'temperature_2m_min',
    #'temperature_2m_max', 
    #'surface_solar_radiation_downwards_sum',
    #'precipitation',
    #'volumetric_soil_water_layer_3',
    #'volumetric_soil_water_layer_4',
    #'soil_temperature_level_4'
]



### Curse of Dimensionality and High Correlation

It does not take much to see that fitting the model purely on all these features will be catastrophic. Many of them explain the exact same data in higher or lower dimensional representations. GMMs rely on distances and covariances across the feature space, making the models sensitive to multicollinearity and high dimensions (relative to the observations).

Our first step is to find the feature pairs that are highly correlated and figure out if we want to (or can) remove them from our feature space.

In [ ]:
corr = X_part2.corr().abs()
select_upper = np.triu(np.ones(corr.shape), k=1).astype(bool)
high_corr = corr.where(select_upper).stack().reset_index()
high_corr.columns = ["feature_1", "feature_2", "correlation"]
high_corr = high_corr[high_corr["correlation"] > 0.7].sort_values("correlation", ascending=False)
high_corr.nlargest(20, "correlation")

Even the first 20 highest correlated features have $\rho>0.95$. This will be highly problematic for the GMM so we need to start pruning. Our first step is to refer back to our feature engineering phase:
1. `ST_PC1` was made to be used _separately_ from `Shallow_mean`, since both explain the same data (just in a different representation).
2. Most 1-day lags and rolls can be removed as it is unlikely to provide much information to the model regarding the temporal behaviour of the crops.
3. The deeper layers of `soil_temperature_level` and `volumetric_soil_water_layer` hardly change as time goes on.
4. Rolling statistics typically hold structural states better than lags.

### Training

We now train our model on the lagged/rolled variables. This process is almost identical to that of our baseline.